# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — connect to the warehouse and rebuild the Week 4 feature/label pipeline

Self-contained on purpose: rebuilds the exact same February (prior-window) features and March-based `decoupling_signature` label from Week 4, so the baseline and the model in this notebook are compared on identical data, not just identical logic.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, numpy as np, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print('Connected.')


In [ ]:
# February (prior-window) page-level features — identical to Week 4
feb_page = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_prior30,
        SUM(gsc_clicks)      AS clicks_prior30,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_sum_position) / SUM(gsc_impressions)
             ELSE NULL END AS avg_position_prior30
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
""").df()
feb_page['ctr_prior30'] = feb_page['clicks_prior30'] / feb_page['impressions_prior30'] * 100

content_meta = con.sql(f"""
    SELECT content_hash_id, content_type, main_intent
    FROM {TABLES['dim_content']}
""").df()
feb_page = feb_page.merge(content_meta, on='content_hash_id', how='left')

mar_page = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_last30,
           SUM(gsc_clicks)      AS clicks_last30
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()

df = feb_page.merge(mar_page, on=['client_hash_id', 'content_hash_id'], how='inner')
df['impr_change_pct'] = 100 * (df['impressions_last30'] - df['impressions_prior30']) / df['impressions_prior30']
df['click_change_pct'] = 100 * (df['clicks_last30'] - df['clicks_prior30']) / df['clicks_prior30'].replace(0, np.nan)
df['decoupling_signature'] = (
    df['impr_change_pct'].between(-10, 10) & (df['click_change_pct'] <= -15)
).astype(int)

# Week 4 baseline rule, recomputed identically (rule doesn't need train/test — it isn't fit)
df['position_bucket'] = pd.cut(df['avg_position_prior30'], bins=[0,3,10,20,50,np.inf],
                                labels=['1-3','4-10','11-20','21-50','51+'], include_lowest=True)
position_ctr_ref = df.groupby('position_bucket', observed=False)['ctr_prior30'].median().rename('position_median_ctr_prior30').reset_index()
df = df.merge(position_ctr_ref, on='position_bucket', how='left')
visibility_threshold = df['impressions_prior30'].median()
df['visible'] = (df['impressions_prior30'] >= visibility_threshold).astype(int)
df['ctr_gap_prior30'] = (df['position_median_ctr_prior30'] - df['ctr_prior30']).clip(lower=0)
df['baseline_score'] = df['visible'] * df['ctr_gap_prior30']

print(f'{len(df):,} rows, positive rate: {df["decoupling_signature"].mean():.4f}')
df.dropna(subset=['avg_position_prior30','content_type','main_intent'], inplace=True)
print(f'{len(df):,} rows after dropping missing content metadata/position')


## 1. Method choice and why

**Question shape:** yes/no with an observed label (`decoupling_signature`) — per the `training-honest-models` skill's own table, that means **Logistic Regression first, then Random Forest**: start readable, add complexity only if it earns its place.

- **Logistic Regression** — a linear, fully inspectable model. If it already beats the Week 4 baseline, that's evidence the pattern is close to linear in these features, and a simple model is preferable to a complex one that isn't meaningfully better (the skill's own "don't reward complexity alone" rule).
- **Random Forest** — allowed to capture non-linear interactions between position, CTR, and content type that a linear model can't. It's the second model specifically so I can see whether the extra complexity is actually earning its keep against Logistic Regression, not just against the baseline.

Both are trained on the **same February-only, prior-window features** as the Week 4 baseline: `impressions_prior30`, `clicks_prior30`, `ctr_prior30`, `avg_position_prior30`, `content_type`, `main_intent`. Nothing from March goes into either model — only into the label used to score all three.

In [ ]:
feature_cols_numeric = ['impressions_prior30', 'clicks_prior30', 'ctr_prior30', 'avg_position_prior30']
feature_cols_categorical = ['content_type', 'main_intent']
print('Numeric features:', feature_cols_numeric)
print('Categorical features:', feature_cols_categorical)
print('Label:', 'decoupling_signature (built from Feb vs. March, never fed to the model as an input)')


## 2. Split design

**Grouped by client (`client_hash_id`), not a plain random split.** Pages from the same client can share patterns — the same site template, the same content team, the same seasonal business cycle — that a plain random split would let leak between train and test, making the model look better than it would on a genuinely new client. `GroupShuffleSplit` guarantees every `client_hash_id` appears in only one side of the split, mirroring the client-holdout validation the lane guide itself recommends and the starter pipeline already used.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print(f'Train rows: {len(train_df):,}  ({train_df["client_hash_id"].nunique()} clients)')
print(f'Test rows:  {len(test_df):,}  ({test_df["client_hash_id"].nunique()} clients)')
print(f'Clients appearing in both train and test: {len(overlap)}  (should be 0)')
print(f'Train positive rate: {train_df["decoupling_signature"].mean():.4f}')
print(f'Test positive rate:  {test_df["decoupling_signature"].mean():.4f}')


## 3. Train + compare vs my baseline

Same test rows, same label, same metric (precision@50 + base rate, matching Week 4) for all three: the frozen Week 4 rule, Logistic Regression, and Random Forest.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

preprocess = ColumnTransformer([
    ('num', StandardScaler(), feature_cols_numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_categorical),
])

X_train = train_df[feature_cols_numeric + feature_cols_categorical]
y_train = train_df['decoupling_signature']
X_test = test_df[feature_cols_numeric + feature_cols_categorical]
y_test = test_df['decoupling_signature']

logreg = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
logreg.fit(X_train, y_train)

rf = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)

print('Both models trained (random_state=42).')


In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_test.mean()
K = 50

scores = {
    'Baseline rule (Week 4)': test_df['baseline_score'].values,
    'Logistic Regression':    logreg.predict_proba(X_test)[:, 1],
    'Random Forest':          rf.predict_proba(X_test)[:, 1],
}

rows = []
for name, s in scores.items():
    rows.append({
        'method': name,
        f'precision@{K}': precision_at_k(s, y_test.values, K),
        'roc_auc': roc_auc_score(y_test, s),
        'lift_vs_base_rate': precision_at_k(s, y_test.values, K) / base_rate,
    })

comparison_table = pd.DataFrame(rows)
print(f'Base rate (test set): {base_rate:.4f}')
comparison_table


## 4. Errors and interpretation

What the stronger model actually leans on, and where it goes wrong — read before the score is believed, per the skill's own instruction.

In [ ]:
# Feature importance from the Random Forest (post-encoding names)
encoded_names = rf.named_steps['prep'].get_feature_names_out()
importances = rf.named_steps['clf'].feature_importances_

importance_df = (
    pd.DataFrame({'feature': encoded_names, 'importance': importances})
    .sort_values('importance', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print('Top features by Random Forest importance:')
importance_df


In [ ]:
# Where the Random Forest is wrong: false positives and false negatives on the test set
test_df = test_df.copy()
test_df['rf_pred_proba'] = rf.predict_proba(X_test)[:, 1]
test_df['rf_pred'] = (test_df['rf_pred_proba'] >= 0.5).astype(int)

false_positives = test_df[(test_df['rf_pred'] == 1) & (test_df['decoupling_signature'] == 0)]
false_negatives = test_df[(test_df['rf_pred'] == 0) & (test_df['decoupling_signature'] == 1)]

print(f'False positives: {len(false_positives):,} ({100*len(false_positives)/len(test_df):.1f}% of test set)')
print(f'False negatives: {len(false_negatives):,} ({100*len(false_negatives)/len(test_df):.1f}% of test set)')
print()
print('False positives tend to have (mean values):')
print(false_positives[feature_cols_numeric].mean().round(2))
print()
print('False negatives tend to have (mean values):')
print(false_negatives[feature_cols_numeric].mean().round(2))
print()
print('3 concrete false-positive examples:')
false_positives[['client_hash_id','content_hash_id'] + feature_cols_numeric + ['rf_pred_proba']].head(3)


**Reading the errors:** *(fill this in once you see your real numbers — a template to reason from, not a number to copy)* if false positives cluster at a much higher `ctr_prior30` or lower `avg_position_prior30` than false negatives, the model may be over-weighting the CTR gap the same way the baseline does, rather than learning something genuinely new. If the top Random Forest feature is something surprising like `content_type` dominating over `ctr_prior30`, sanity-check it — a feature that seems to matter far more than domain sense would suggest is worth treating as a leakage flag, not a discovery, per the skill's own warning ("suspiciously perfect = probably leakage"). Since every feature here is Feb-only and the label is Feb-vs-March, this result should NOT look suspiciously perfect — if `roc_auc` came back above ~0.95, that's a signal to re-check the feature list before trusting it, not a result to celebrate.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.